<div class="blog-language-switch" role="group" aria-label="Article language"><span aria-current="page">English</span><a href="/ipynb/zh-CN/Computer-Science/Computer-Organization/04-sequential-logic-and-state.html" lang="zh-CN" hreflang="zh-CN">中文</a></div>

[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Sequential Logic and State** {#sequential-logic-and-state}

Chapter 03 built combinational circuits: once their inputs settle, their outputs are determined by the current input values alone. An adder can calculate a sum, but it cannot remember that sum after its operands disappear. A processor needs more. It must remember the current instruction address, intermediate results, condition flags, outstanding control decisions, and millions or billions of data bits.

**Sequential logic** adds this memory. It combines storage elements with combinational logic so that a system can move through a sequence of states. The clock does not perform the computation; it creates agreed moments at which storage elements accept the results produced during the preceding interval. This separation turns a continuously changing electrical network into a machine whose behavior can be reasoned about cycle by cycle.

### **Why Digital Systems Need State** {#why-digital-systems-need-state}

A **state** is the smallest remembered information needed to determine future behavior. It is not necessarily the complete history. A traffic-light controller does not need a log of every car that has passed; it may need only the current phase, an elapsed-time condition, and whether a pedestrian request is pending. A processor similarly represents its architectural history compactly through values such as the program counter and registers.

A purely combinational system follows

$$
Y=F(X),
$$

where $X$ is the current input vector, $F$ is a Boolean or arithmetic function, and $Y$ is the resulting output vector. If the same $X$ is applied twice, the same $Y$ appears both times.

A sequential system instead follows a state-transition model:

$$
S[k+1]=F(S[k],X[k]),
$$

$$
Y[k]=G(S[k],X[k]).
$$

- $k$ identifies the current discrete step or clock cycle.
- $S[k]$ is the state stored during cycle $k$.
- $X[k]$ is the external input observed during that cycle.
- $F$ is the **next-state function**: it decides what should be remembered next.
- $S[k+1]$ is captured state for the following cycle.
- $G$ is the **output function**. It may depend on state alone or on both state and current input.

The same input can therefore produce different outputs when the stored state differs. Pressing a play/pause button is a simple analogy: the button event is identical, but the next action depends on whether the player is currently playing or paused.

State solves several recurring hardware problems:

| Need | State that is retained | Example |
|---|---|---|
| continue a computation | partial result and progress | iterative multiply or divide |
| preserve a value | an $n$-bit word | processor register |
| choose the next action | control state | instruction controller |
| count events | binary count | timer or performance counter |
| remember an address | current location | program counter |
| buffer communication | queued data and pointers | FIFO between components |

State also introduces new obligations. The system needs a known initial condition, rules for when state may change, and timing guarantees that prevent storage from sampling unstable data. A bug in combinational logic produces a wrong function. A bug in sequential logic may appear only after a particular history, which makes disciplined state design essential.

<details>
<summary>Python analogy: the same input behaves differently when state is retained</summary>

~~~python
class Accumulator:
    """A tiny state machine that remembers a running total."""

    def __init__(self) -> None:
        self.state = 0

    def clock(self, value: int, enable: bool = True) -> int:
        # First calculate next state from current state and input.
        next_state = self.state + value if enable else self.state

        # The assignment models capture at a clock edge.
        self.state = next_state
        return self.state


acc = Accumulator()
assert acc.clock(5) == 5
assert acc.clock(5) == 10       # Same input, different current state.
assert acc.clock(100, False) == 10
assert acc.clock(-3) == 7
~~~

</details>

### **Clocks and Synchronous Design** {#clocks-and-synchronous-design}

A **clock** is a periodic signal used to coordinate state updates. Its important events are usually rising edges, falling edges, or both. In a positive-edge-triggered design, registers observe their inputs at each rising edge and hold the captured values until the next rising edge.

If the clock period is $T_{clk}$, the nominal frequency is

$$
f_{clk}=\frac{1}{T_{clk}}.
$$

$T_{clk}$ is measured in seconds per cycle and $f_{clk}$ in cycles per second, or hertz. A 2 ns period corresponds to $1/(2\times10^{-9})=500$ MHz. A higher frequency creates more state-update opportunities per second, but it also gives combinational logic less time to settle and usually increases clock-distribution power.

![Current state and input pass through combinational logic; the next clock edge captures the result and begins another cycle.](assets/synchronous-state-cycle.svg){fig-align="center" width="100%"}

The most common synchronous datapath has three conceptual parts:

1. A **launch register** exposes state captured at the previous edge.
2. Combinational logic transforms that state and current inputs.
3. A **capture register** stores the settled result at the next edge.

All registers appear to update together, although physical clock edges reach them at slightly different times. Crucially, a register's new output cannot race through the entire design and be captured on the same ideal edge: its output changes only after a clock-to-Q delay, while the receiving register has already sampled that edge. This creates clean cycle boundaries.

The clock is coordination, not causation. Combinational gates continuously react whenever inputs change. The clock merely decides when their result becomes durable state. Clocking every logic gate would be wasteful; designers place registers at meaningful stage boundaries and allow logic to operate between them.

A synchronous design is attractive because timing can be reduced to local register-to-register checks. The alternative, **asynchronous design**, allows components to communicate through handshakes and local completion rather than one global rhythm. Asynchronous circuits can avoid unnecessary waiting and reduce clock power, but their control and verification are more specialized. Modern processors are predominantly synchronous while still using asynchronous interfaces and multiple clock domains.

Reset establishes a known initial state. A **synchronous reset** changes state only on the active clock edge; an **asynchronous reset** can force a storage element immediately. Asynchronous assertion is useful when a clock is unavailable, but reset release must be synchronized so different registers do not resume on inconsistent edges.

| Design rule | Why it matters |
|---|---|
| use one defined active edge per domain | avoids ambiguous sampling relationships |
| keep ordinary data away from clock pins | data glitches must not become extra clock events |
| cross clock domains deliberately | unrelated clocks do not share setup/hold guarantees |
| define reset values and release behavior | prevents unknown startup state |
| analyze both maximum and minimum path delay | setup and hold are different constraints |

### **Latches and Flip-Flops** {#latches-and-flip-flops}

Latches and flip-flops are circuits that preserve one bit through feedback. Their external behavior is digital, but the storage mechanism is physical: cross-coupled gain reinforces one of two stable voltage states. The two output labels are commonly $Q$ and $\overline Q$, where the bar denotes logical complement during valid operation.

The key distinction is **when** an input may affect stored state:

- a latch is **level-sensitive** and may be transparent throughout an enabled interval;
- a flip-flop is **edge-triggered** and samples near one clock transition.

#### **SR Latches** {#sr-latches}

The **set-reset latch** is the simplest explicit storage element. Two cross-coupled NOR gates create feedback: each output helps determine the other gate's input. Once one output becomes 1, the feedback keeps that condition after the initiating input returns to 0.

![A cross-coupled NOR SR latch uses feedback to preserve one of two stable states.](assets/sr-latch.svg){fig-align="center" width="82%"}

For the active-high NOR implementation shown above:

| $S$ | $R$ | $Q_{next}$ | Meaning |
|---:|---:|---:|---|
| 0 | 0 | $Q$ | hold the previous state |
| 1 | 0 | 1 | set |
| 0 | 1 | 0 | reset |
| 1 | 1 | invalid | both outputs are forced low |

When $S=R=0$, neither input requests a change; feedback reproduces the previous $Q$. When $S=1$, the lower NOR output becomes 0, which permits the upper output $Q$ to become 1. Reset is symmetric.

The $S=R=1$ case is not merely a logically inconvenient row. Both outputs become 0, violating the expectation that $Q$ and $\overline Q$ are complements. If both inputs then return to 0 nearly together, tiny physical delay differences decide which stable state wins. A design that allows this case has given up deterministic digital behavior.

NAND-based SR latches are also common, but their external inputs are active-low. The truth table must therefore be read with the circuit polarity, not memorized without context. A bubble or overbar on a schematic indicates that asserting a control means driving it to 0.

An SR latch explains the origin of memory, but two independent data commands are awkward for ordinary synchronous storage. The D latch removes the invalid command by deriving set and reset from one data input.

<details>
<summary>Python state model: exercise valid and invalid SR-latch commands</summary>

~~~python
class SRLatch:
    def __init__(self, initial_q: int = 0) -> None:
        self.q = int(bool(initial_q))

    def apply(self, set_input: int, reset_input: int) -> int:
        set_input = int(bool(set_input))
        reset_input = int(bool(reset_input))

        if set_input and reset_input:
            # Real hardware can lose deterministic state when both are released.
            raise ValueError("S=R=1 is invalid for an active-high NOR latch")
        if set_input:
            self.q = 1
        elif reset_input:
            self.q = 0
        # S=R=0 performs no assignment and therefore holds state.
        return self.q


latch = SRLatch()
assert latch.apply(1, 0) == 1
assert latch.apply(0, 0) == 1
assert latch.apply(0, 1) == 0
~~~

</details>

#### **D Latches** {#d-latches}

A **D latch** has a data input $D$ and an enable input $E$. While $E=1$, the latch is transparent and $Q$ follows $D$ after gate delay. When $E=0$, the feedback path preserves the last value. Its next-state equation is

$$
Q^{+}=ED+\overline E\,Q.
$$

- $Q^{+}$ means the value after the latch has responded.
- $D$ is the new data candidate.
- $E$ is the enable level.
- $ED$ passes $D$ when $E=1$.
- $\overline E\,Q$ feeds back the old state when $E=0$.

The two product terms cannot both be active: $E$ and $\overline E$ are complements. The equation is therefore a 2-to-1 multiplexer whose select is $E$, with new data on one input and current state on the other.

![A transparent D latch derives complementary set/reset controls from one data input and an enable signal.](assets/d-transparent-latch.svg){fig-align="center" width="66%"}

*Image source: [D-Type Transparent Latch.svg](https://commons.wikimedia.org/wiki/File:D-Type_Transparent_Latch.svg), Inductiveload, public domain.*

Transparency is useful but demands care. If several active-high latches share the same enable, a change can pass through more than one stage during that interval, causing a **race-through** problem. Two-phase latch systems avoid this by alternating non-overlapping phases: one group is transparent while the other holds. They can also support **time borrowing**, where a slow block uses part of a neighboring phase, but verification is more complex than simple edge-to-edge timing.

Latches are smaller and can be efficient in custom high-performance designs. Edge-triggered flip-flops are more common in introductory register-transfer reasoning because the sampling instant is easier to define.

| Enable $E$ | Data $D$ | Latch response |
|---:|---:|---|
| 0 | any | hold $Q$ |
| 1 | 0 | drive $Q$ toward 0 |
| 1 | 1 | drive $Q$ toward 1 |

The word 鈥渢ransparent鈥?does not mean zero delay. It means changes are allowed to propagate while enabled. Real gate delay still determines when $Q$ moves and whether a late data transition satisfies the closing-edge timing requirement.

#### **Edge-Triggered Flip-Flops** {#edge-triggered-flip-flops}

An **edge-triggered D flip-flop** captures $D$ only around an active clock edge. Between edges, changes on $D$ do not alter $Q$. For an ideal positive-edge device,

$$
Q[k+1]=D\big|_{\text{rising edge }k+1}.
$$

The expression says that next-cycle output equals the data value sampled at rising edge $k+1$. It does not say the output changes instantly: real $Q$ appears after a clock-to-Q delay.

![A positive-edge-triggered D flip-flop uses internal feedback and gating so the external output changes only after an active edge.](assets/edge-triggered-d-flip-flop.svg){fig-align="center" width="54%"}

*Image source: [Edge triggered D flip flop.svg](https://commons.wikimedia.org/wiki/File:Edge_triggered_D_flip_flop.svg), Nolanjshettle, CC BY-SA 3.0.*

One conceptual implementation uses two latches with opposite transparency, often called **master** and **slave**. Before the active edge, the first latch tracks input while the second holds output. At the edge their roles exchange: the first closes on the final input value and the second opens to publish that captured value. Practical standard cells may use pulse-triggered or optimized transistor structures, but the external timing contract remains edge sampling.

A flip-flop data sheet or timing library specifies more than a truth table:

- **setup time:** how long $D$ must be stable before the edge;
- **hold time:** how long $D$ must remain stable after the edge;
- **clock-to-Q propagation delay:** latest time at which new $Q$ is guaranteed;
- **clock-to-Q contamination delay:** earliest time at which $Q$ might begin changing;
- asynchronous reset or set behavior, if present.

These values depend on voltage, temperature, manufacturing process, input transition rate, and output load. A simulator that treats a flip-flop as an instantaneous assignment is useful for functional logic but cannot prove physical timing correctness.

| Storage element | Sensitive interval | Main strength | Main concern |
|---|---|---|---|
| SR latch | whenever set/reset act | exposes feedback principle | forbidden command |
| D latch | entire enabled level | compact; permits time borrowing | transparency and race-through |
| D flip-flop | narrow edge window | simple cycle-by-cycle model | clock power and edge timing |

### **Registers, Counters, and Shift Registers** {#registers-counters-and-shift-registers}

An $n$-bit **register** places $n$ flip-flops side by side with a shared clock. Each flip-flop stores one bit, so all bits of a word are captured on the same logical edge. Registers hold processor operands, instruction fields, addresses, status flags, and pipeline boundaries.

A register often needs a load enable. Rather than gating the clock with ordinary logic, a multiplexer selects either new data or feedback for each bit:

$$
D_i=Load\cdot X_i+\overline{Load}\cdot Q_i.
$$

$i$ identifies one bit position, $X_i$ is incoming data, and $Q_i$ is the current stored bit. When `Load=1`, $D_i=X_i$; when `Load=0`, $D_i=Q_i$, so the next edge recaptures the same value. Dedicated clock-gating cells can save dynamic power, but they are designed to prevent glitches and are inserted under controlled rules.

A **counter** is a register whose next-state logic performs a progression. An unsigned modulo-$2^n$ up-counter follows

$$
Q[k+1]=(Q[k]+1)\bmod 2^n.
$$

$n$ is the register width, and modulo $2^n$ discards carry beyond the top bit. A 3-bit counter therefore cycles `000, 001, ..., 111, 000`. A **ripple counter** lets one flip-flop output clock the next; it is small but transitions ripple at different times and create temporary codes. A **synchronous counter** gives every flip-flop the same clock and computes all next bits combinationally, making its timing easier to control.

A **shift register** connects each stage's next input to a neighboring stage. It can convert serial data to parallel words, delay a bit stream, perform serial communication, and implement compact pattern generators. Common modes include hold, parallel load, shift left, and shift right.

::: {.diagram-scroll}
![A three-bit bidirectional shift register uses multiplexing around positive-edge-triggered D flip-flops to choose left- or right-moving data.](assets/bidirectional-shift-register.svg){fig-align="center" width="100%"}
:::

*Image source: [3 bit bi-directional shift register circuit.svg](https://commons.wikimedia.org/wiki/File:3_bit_bi-directional_shift_register_circuit.svg), Dizquierdol, CC0 1.0.*

For a logical right shift of an $n$-bit register, stage $i$ receives old bit $i+1$ and the most-significant stage receives a serial input, often 0. Because all stages capture together, each uses the **old** neighbor value. Updating a software list in place from left to right would not model that parallel behavior correctly; next state must be computed before current state is replaced.

| Structure | Next-state operation | Typical use |
|---|---|---|
| parallel register | load or hold one word | operands and pipeline state |
| counter | increment, decrement, or reload | timers, addresses, event counts |
| shift register | move neighbor bits | serialization and delays |
| ring counter | rotate one-hot state | simple phase sequencing |
| linear-feedback shift register | shift plus XOR feedback | test patterns and pseudo-random sequences |

<details>
<summary>Python model: parallel register, modulo counter, and bidirectional shift register</summary>

~~~python
class SequentialWord:
    def __init__(self, width: int, initial: int = 0) -> None:
        if width <= 0:
            raise ValueError("width must be positive")
        self.width = width
        self.mask = (1 << width) - 1
        self.q = initial & self.mask

    def load(self, value: int, enable: bool = True) -> int:
        # Compute the D inputs first, then model one simultaneous edge.
        next_q = value & self.mask if enable else self.q
        self.q = next_q
        return self.q

    def count_up(self, enable: bool = True) -> int:
        next_q = (self.q + 1) & self.mask if enable else self.q
        self.q = next_q
        return self.q

    def shift(self, direction: str, serial_in: int = 0) -> int:
        serial_in &= 1
        old_q = self.q

        if direction == "right":
            next_q = (old_q >> 1) | (serial_in << (self.width - 1))
        elif direction == "left":
            next_q = ((old_q << 1) & self.mask) | serial_in
        else:
            raise ValueError("direction must be left or right")

        self.q = next_q
        return self.q


counter = SequentialWord(width=3, initial=0b110)
assert counter.count_up() == 0b111
assert counter.count_up() == 0b000   # Modulo-8 wraparound.

shift_register = SequentialWord(width=4, initial=0b1011)
assert shift_register.shift("right", serial_in=0) == 0b0101
assert shift_register.shift("left", serial_in=1) == 0b1011
~~~

</details>

### **Finite-State Machines** {#finite-state-machines}

A **finite-state machine** (FSM) is a mathematical and hardware model for control whose relevant history can be represented by a finite set of states. The state register remembers the current abstract situation; combinational next-state logic examines that state and inputs; output logic generates control signals.

An FSM is useful whenever behavior is described by phases or protocols: fetching and executing an instruction, recognizing a bit pattern, controlling a cache miss, sequencing a memory request, or parsing a communication packet. It is less suitable for storing large arbitrary data because each additional state bit doubles the potential encoded state space.

Formally, a deterministic FSM contains:

$$
S[k+1]=\delta(S[k],X[k]),
$$

where $\delta$ is the transition function, and an output function $\lambda$. Every legal pair of current state and input must identify one next state. Missing transitions become accidental behavior in simulation or unintended hardware defaults.

#### **Moore and Mealy Machines** {#moore-and-mealy-machines}

The two standard output formulations differ in whether current input has a direct path to output.

For a **Moore machine**,

$$
Y[k]=\lambda(S[k]).
$$

Output depends only on registered state. It therefore changes after a state update and is naturally stable between edges, apart from state-register and decode delay.

For a **Mealy machine**,

$$
Y[k]=\lambda(S[k],X[k]).
$$

Output can react during the current cycle as input changes. This often needs fewer states and can respond one cycle earlier, but input glitches or late transitions may reach output unless the output is registered or carefully constrained.

![A Moore output is decoded from state, whereas a Mealy output also has a direct combinational input path.](assets/fsm-moore-mealy.svg){fig-align="center" width="100%"}

Consider detecting the serial pattern `101` with overlap. A Mealy detector needs states representing useful suffixes: `A` means no useful suffix, `B` means the latest suffix is `1`, and `C` means it is `10`. While in `C`, receiving `1` immediately asserts detection and returns to `B` because that final `1` may begin another match. A Moore detector normally adds a dedicated detected state whose output is 1, so the visible pulse occurs after entering that state.

| Property | Moore | Mealy |
|---|---|---|
| output depends on | state | state and current input |
| response | usually after a state edge | may occur within current cycle |
| state count | can be larger | often smaller |
| output stability | easier to reason about | sensitive to input timing/glitches |
| common use | registered control signals | compact protocol decisions |

Neither model is universally superior. A practical design may calculate a Mealy condition but register it before it leaves the module, combining fast internal detection with a clean external interface.

#### **State Transition Design** {#state-transition-design}

FSM design should begin with behavior rather than binary encodings. A dependable process is:

1. **Define inputs, outputs, reset, and clock domain.** State signal polarity and what counts as one event.
2. **Identify the minimum relevant history.** Give each distinguishable situation a meaningful state name.
3. **Draw every transition.** Label each edge with the input condition and, for Mealy outputs, the output action.
4. **Build a transition table.** Enumerate current state and legal input combinations.
5. **Choose an encoding.** Assign bit patterns to abstract states.
6. **Derive next-state and output logic.** Use Boolean simplification or synthesis tools.
7. **Define illegal-state recovery.** Reset or redirect unused codes rather than assuming they never occur.
8. **Verify sequences and invariants.** Test normal paths, reset, overlapping events, and unexpected inputs.

For the `101` Mealy detector:

| Current state | Input 0: next/output | Input 1: next/output | Remembered suffix |
|---|---|---|---|
| A | A / 0 | B / 0 | none |
| B | C / 0 | B / 0 | `1` |
| C | A / 0 | B / 1 | `10` |

State encoding changes hardware organization without changing abstract behavior:

- **binary encoding** uses $\lceil\log_2N\rceil$ flip-flops for $N$ states but can require more decode logic;
- **one-hot encoding** uses $N$ flip-flops with exactly one asserted, often simplifying transitions and improving speed in FPGA fabrics;
- **Gray-like encoding** tries to change one bit on important transitions, reducing simultaneous switching and decode hazards.

The smallest number of flip-flops does not automatically yield the smallest or fastest circuit. One-hot designs use more state bits but may remove deep Boolean logic. Synthesis tools can explore encodings, yet meaningful state names and explicit behavior remain essential for verification.

<details>
<summary>Python implementation: overlapping Mealy detector for the serial pattern 101</summary>

~~~python
from enum import Enum, auto


class DetectorState(Enum):
    NONE = auto()       # No useful suffix.
    SAW_1 = auto()      # Latest useful suffix is "1".
    SAW_10 = auto()     # Latest useful suffix is "10".


class Pattern101Detector:
    def __init__(self) -> None:
        self.state = DetectorState.NONE

    def clock(self, bit: int) -> int:
        """Return the Mealy output produced by this input and transition."""
        if bit not in (0, 1):
            raise ValueError("input must be one bit")

        output = 0
        if self.state is DetectorState.NONE:
            next_state = DetectorState.SAW_1 if bit else DetectorState.NONE
        elif self.state is DetectorState.SAW_1:
            next_state = DetectorState.SAW_1 if bit else DetectorState.SAW_10
        else:  # SAW_10
            output = int(bit == 1)  # Completing 101 is a Mealy action.
            next_state = DetectorState.SAW_1 if bit else DetectorState.NONE

        # All next-state reasoning used the old state; capture happens last.
        self.state = next_state
        return output


detector = Pattern101Detector()
stream = [1, 0, 1, 0, 1, 1]
outputs = [detector.clock(bit) for bit in stream]
assert outputs == [0, 0, 1, 0, 1, 0]  # Detects two overlapping matches.
~~~

</details>

The code is a behavioral model, not gate-level hardware. Its ordering is nevertheless important: decide output and `next_state` from old state, then assign state once. Hardware-description languages use nonblocking assignments for the same reason in clocked processes.

### **Register Files** {#register-files}

A **register file** is a small, fast addressed collection of registers. Individual registers provide storage; decoders and multiplexers turn them into a structure that can select operands by register number. A typical scalar processor needs two source operands and one destination per instruction, so a common organization has two read ports and one write port, abbreviated **2R1W**.

![Two independent read multiplexers select source operands, while a decoder enables one destination register for a clocked write.](assets/register-file-ports.svg){fig-align="center" width="100%"}

For $N$ registers of $W$ bits:

- capacity is $N\times W$ bits;
- each address needs $\lceil\log_2N\rceil$ bits;
- a write decoder converts the destination address to $N$ one-hot enables;
- each read port needs selection hardware able to choose one of $N$ words.

For example, 32 registers require 5-bit addresses because $2^5=32$. A 32-register, 64-bit file stores 2048 data bits, but its area is substantially more than 2048 isolated bit cells because read and write ports add wires, access devices, decoders, and multiplexing. Port count is expensive: multiple instructions per cycle may require many read and write ports, so superscalar designs often use banking, replication, bypass networks, or physical register files with carefully engineered access structures.

Writes are normally synchronous: `write_enable`, address, and data must satisfy timing around the active edge. Reads may be combinational, producing data after address decode and MUX delay, or synchronous, returning data on a later edge. The choice changes the datapath schedule.

A same-cycle read and write to one address needs an explicit contract:

- **read-first:** the read sees the old value;
- **write-first:** the read sees new write data;
- **no-change or undefined:** the interface promises neither.

Processor pipelines often use **bypassing** to forward a pending result directly rather than waiting for it to be written and then read. This is not a property to guess from a generic register-file diagram; it is an organizational decision defined by the design.

<details>
<summary>Python model: a two-read, one-write register file with edge commit</summary>

~~~python
class RegisterFile:
    def __init__(self, register_count: int, width: int, hardwired_zero: bool = False):
        if register_count <= 0 or width <= 0:
            raise ValueError("register count and width must be positive")
        self.values = [0] * register_count
        self.mask = (1 << width) - 1
        self.hardwired_zero = hardwired_zero
        self.pending_write: tuple[int, int] | None = None

    def read2(self, address_a: int, address_b: int) -> tuple[int, int]:
        # Combinational read: no state changes here.
        return self.values[address_a], self.values[address_b]

    def request_write(self, address: int, data: int, enable: bool = True) -> None:
        # Inputs are prepared during the cycle; the array is not changed yet.
        self.pending_write = (address, data & self.mask) if enable else None

    def rising_edge(self) -> None:
        # One destination is updated when the active edge arrives.
        if self.pending_write is not None:
            address, data = self.pending_write
            if not (self.hardwired_zero and address == 0):
                self.values[address] = data
        self.pending_write = None


rf = RegisterFile(register_count=8, width=16, hardwired_zero=True)
rf.request_write(3, 0x12345)
assert rf.read2(3, 0) == (0, 0)       # Read-first before the edge.
rf.rising_edge()
assert rf.read2(3, 0) == (0x2345, 0)  # Width mask and hardwired R0.
~~~

</details>

### **Memory Building Blocks** {#memory-building-blocks}

Registers scale poorly when a system needs many words. A memory array arranges bit cells in rows and columns so address decoding and sensing hardware can be shared. An address selects a word; data lines carry a full word; control signals distinguish reading, writing, enabling, and sometimes refreshing.

If a memory has $A$ address bits and each word has $W$ bits, its ideal logical capacity is

$$
C=2^A\times W\text{ bits}.
$$

$2^A$ is the number of selectable addresses and $W$ is the number of data bits per address. Physical capacity also includes redundancy, error-correcting bits, decoders, sense amplifiers, drivers, and control circuits.

![ROM, SRAM, and DRAM store bits with different physical mechanisms but all scale through addressed arrays and shared access circuitry.](assets/memory-building-blocks.svg){fig-align="center" width="100%"}

#### **ROM** {#rom}

**Read-only memory** stores a mapping from address to word that ordinary operation does not modify. Conceptually it is a combinational lookup table:

$$
Data=ROM[Address].
$$

A decoder activates one row, and programmed connections on bit lines determine the returned pattern. ROM is useful for boot code, fixed microcode, lookup tables, device configuration, and constants because its contents survive ordinary power cycles in nonvolatile implementations.

鈥淩ead-only鈥?describes the normal interface, not one universal fabrication method:

- **mask ROM** is fixed during manufacturing;
- **PROM** is programmed once after manufacture;
- **EPROM** can be erased with ultraviolet light and rewritten;
- **EEPROM** and **flash** are electrically erasable nonvolatile memories, usually written with slower, coarser operations than reads.

A small Boolean function can be implemented either as minimized gates or as ROM contents. A ROM avoids deriving special-purpose equations and is easy to update when programmable, but it stores every addressed output pattern and may be less efficient than optimized logic for sparse simple functions.

<details>
<summary>Python model: immutable ROM lookup with address validation</summary>

~~~python
class ROM:
    def __init__(self, words: list[int], width: int) -> None:
        self._mask = (1 << width) - 1
        self._words = tuple(word & self._mask for word in words)

    def read(self, address: int) -> int:
        if not 0 <= address < len(self._words):
            raise IndexError("ROM address out of range")
        return self._words[address]


# A four-entry lookup table returning seven-segment-like patterns.
rom = ROM([0b0111111, 0b0000110, 0b1011011, 0b1001111], width=7)
assert rom.read(2) == 0b1011011
~~~

</details>

#### **SRAM** {#sram}

**Static random-access memory** stores each bit in a bistable feedback circuit. A conventional six-transistor, or 6T, cell uses two cross-coupled inverters to hold $Q$ and $\overline Q$, plus two access transistors controlled by a word line. It is 鈥渟tatic鈥?because the value remains without periodic refresh while power is present; it is not nonvolatile.

During a read, complementary bit lines are commonly precharged. Activating the word line connects the cell to those bit lines, and the stored state creates a small differential voltage that a sense amplifier expands into a full digital value. During a write, strong bit-line drivers force complementary levels and overpower the old cell state.

The read must not accidentally flip the cell, while the write must be strong enough to change it. Transistor sizing therefore balances **read stability**, **write ability**, leakage, speed, and area. These analog details sit below the clean digital abstraction of `read(address)` and `write(address, data)`.

SRAM is faster and avoids refresh, but a 6T cell occupies more area than a DRAM cell. It is therefore used where latency matters more than density: processor caches, small on-chip memories, queues, and lookup structures. Register files may use SRAM-like custom cells, but their multiport and timing requirements make them a distinct organization.

| SRAM operation | Physical action | Digital result |
|---|---|---|
| hold | word line off; feedback reinforces state | bit remains while powered |
| read | word line on; cell perturbs precharged bit lines | sense amplifier returns 0 or 1 |
| write | bit-line drivers impose new complementary levels | feedback settles to new state |

Unlike a register, an SRAM array shares access circuitry across many words. That sharing improves density but adds address decode, word-line, bit-line, and sensing delays.

#### **DRAM** {#dram}

**Dynamic random-access memory** stores a bit as charge on a tiny capacitor, usually accessed through one transistor. A charged and discharged capacitor represent the two logical states, although the exact voltage interpretation and sensing method depend on the technology.

The word line enables the access transistor and connects the capacitor to a bit line. Because the stored charge is tiny, a read shares charge with the bit line and a sense amplifier decides which state was present. This process disturbs the original capacitor level, so the row must be restored after reading. Charge also leaks even when untouched, which requires periodic **refresh**.

DRAM arrays exploit geometry for density. A row address activates many cells at once into a **row buffer** of sense amplifiers. A later column selection chooses requested words from that open row. Access to another column in the same row can be relatively fast; changing rows requires precharge and activation work. This is why physical address mapping, locality, and memory-controller scheduling affect performance.

The single-transistor-plus-capacitor cell is much denser than a 6T SRAM cell, making DRAM suitable for main memory. Its disadvantages are greater access latency, refresh overhead, destructive-read restoration, and more complex control.

| Property | Register | SRAM | DRAM | ROM / flash-like storage |
|---|---|---|---|---|
| basic storage | flip-flop/latch | bistable cell | capacitor charge | programmed/nonvolatile cell |
| retains without power | no | no | no | generally yes |
| periodic refresh | no | no | yes | no |
| relative latency | lowest | low | higher | technology-dependent |
| relative density | lowest | medium | high | high |
| common role | immediate CPU state | cache/on-chip arrays | main memory | boot and persistent constants |

<details>
<summary>Python abstraction: contrast SRAM-like access with DRAM refresh bookkeeping</summary>

~~~python
class SRAMModel:
    def __init__(self, word_count: int, width: int) -> None:
        self.mask = (1 << width) - 1
        self.words = [0] * word_count

    def read(self, address: int) -> int:
        return self.words[address]

    def write(self, address: int, data: int) -> None:
        self.words[address] = data & self.mask


class DRAMModel(SRAMModel):
    """Functional data plus a simplified refresh-age constraint."""

    def __init__(self, word_count: int, width: int, refresh_limit: int) -> None:
        super().__init__(word_count, width)
        self.age = [0] * word_count
        self.refresh_limit = refresh_limit

    def tick(self) -> None:
        # Real controllers refresh rows; this model ages individual words.
        self.age = [value + 1 for value in self.age]

    def refresh(self, address: int) -> None:
        self.age[address] = 0

    def read(self, address: int) -> int:
        if self.age[address] > self.refresh_limit:
            raise RuntimeError("retention guarantee expired before refresh")
        value = super().read(address)
        self.refresh(address)  # Model restore after destructive sensing.
        return value

    def write(self, address: int, data: int) -> None:
        super().write(address, data)
        self.refresh(address)


dram = DRAMModel(word_count=4, width=8, refresh_limit=3)
dram.write(1, 0xAB)
for _ in range(3):
    dram.tick()
assert dram.read(1) == 0xAB
assert dram.age[1] == 0
~~~

</details>

The model is intentionally behavioral. Real DRAM does not count an integer age beside each word; a controller schedules row refresh commands according to a timing specification. The code makes the obligation visible without pretending to simulate capacitor voltages.

### **Timing Constraints** {#timing-constraints}

Functional correctness says which value a circuit should produce after signals settle. **Timing correctness** says that the value becomes stable early enough, remains stable long enough, and is sampled by the intended clock edge. A circuit can pass every zero-delay truth-table test and still fail in silicon because its path is too slow or too fast.

Timing analysis follows paths from a launching storage element, through combinational logic and wires, to a capturing storage element. Maximum delays limit frequency; minimum delays prevent new data from arriving too soon. Both checks are required for every relevant process, voltage, temperature, and clock condition.

#### **Propagation and Contamination Delay** {#propagation-and-contamination-delay}

**Propagation delay** is a maximum: after an input event, when is the output guaranteed to have reached its correct new value? For a gate,

$$
t_{pd}=\max(t_{PLH},t_{PHL}),
$$

where $t_{PLH}$ is low-to-high delay and $t_{PHL}$ is high-to-low delay. The larger direction gives a conservative guarantee.

**Contamination delay** is a minimum: how soon might the old output begin to change? It is commonly written

$$
t_{cd}=\min(t_{cd,LH},t_{cd,HL}).
$$

The two concepts bound an uncertainty interval. Before $t_{cd}$, the old output is guaranteed unchanged. After $t_{pd}$, the new correct output is guaranteed. Between them, the output may be transitioning or glitching.

For a multi-gate path, latest arrival follows the slowest predecessor through maximum delays, while earliest arrival follows the earliest predecessor through minimum delays. Wire delay, fan-out, and loading must be included; simply counting gates can miss the physical critical path.

| Delay quantity | Question answered | Main use |
|---|---|---|
| $t_{cq,max}$ | latest new register output after clock | setup analysis |
| $t_{cq,min}$ | earliest register output disturbance | hold analysis |
| $t_{comb,max}$ | latest combinational result | setup analysis |
| $t_{comb,min}$ | earliest combinational change | hold analysis |
| $t_{setup}$ | required stability before capture edge | setup analysis |
| $t_{hold}$ | required stability after capture edge | hold analysis |

Adding buffers can improve a hold violation by increasing minimum delay, even though it makes the path slower. Increasing the clock period can repair a setup violation but cannot repair a same-edge hold violation. This difference is one of the most important timing intuitions in synchronous design.

#### **Setup and Hold Time** {#setup-and-hold-time}

A capture flip-flop needs its input stable around the active edge. **Setup time** is the required stable interval before that edge; **hold time** is the required interval after it. Together they create a no-change window at the flip-flop input.

![Data must be stable throughout the setup interval before the capture edge and the hold interval after it.](assets/setup-hold-window.svg){fig-align="center" width="100%"}

Define clock skew as

$$
t_{skew}=t_{capture}-t_{launch},
$$

so positive skew means the capture clock edge arrives later than the launch edge. The setup constraint for one register-to-register path is

$$
T_{clk}\ge t_{cq,max}+t_{comb,max}+t_{setup}-t_{skew}+t_{uncertainty}.
$$

- $T_{clk}$ is the period between corresponding launch and capture cycles.
- $t_{cq,max}$ is the latest launch-register output response.
- $t_{comb,max}$ is the maximum delay through logic and wiring.
- $t_{setup}$ is the capture register's setup requirement.
- positive $t_{skew}$ gives data more time and therefore appears with a minus sign.
- $t_{uncertainty}$ reserves margin for jitter and modeling variation.

Suppose $t_{cq,max}=0.12$ ns, $t_{comb,max}=1.55$ ns, $t_{setup}=0.10$ ns, $t_{skew}=0.04$ ns, and uncertainty is 0.08 ns. The period must be at least

$$
0.12+1.55+0.10-0.04+0.08=1.81\text{ ns},
$$

which corresponds to at most approximately $1/1.81\text{ ns}=552$ MHz for this path.

The hold constraint uses earliest arrival:

$$
t_{cq,min}+t_{comb,min}\ge t_{hold}+t_{skew}+t_{uncertainty,hold}.
$$

The left side is how soon new data can reach the capture input. The right side is how long the old data must remain after the delayed capture edge. Positive skew now makes the requirement harder. If new data arrives too soon, increasing $T_{clk}$ does nothing because both launch and capture concern the same edge. The path needs added minimum delay, adjusted clock distribution, a different cell, or architectural restructuring.

Timing **slack** is requirement minus arrival for setup, or available minimum delay minus required minimum delay for hold. Positive slack passes; negative slack quantifies a violation. Sign conventions in commercial reports vary, so always read the tool's definition rather than copying a formula blindly.

<details>
<summary>Python timing checker: calculate setup and hold slack for one path</summary>

~~~python
from dataclasses import dataclass


@dataclass(frozen=True)
class TimingPath:
    clock_period: float
    clock_to_q_max: float
    clock_to_q_min: float
    combinational_max: float
    combinational_min: float
    setup: float
    hold: float
    skew: float = 0.0
    setup_uncertainty: float = 0.0
    hold_uncertainty: float = 0.0

    def setup_slack(self) -> float:
        required_period = (
            self.clock_to_q_max
            + self.combinational_max
            + self.setup
            - self.skew
            + self.setup_uncertainty
        )
        return self.clock_period - required_period

    def hold_slack(self) -> float:
        earliest_arrival = self.clock_to_q_min + self.combinational_min
        required_stability = self.hold + self.skew + self.hold_uncertainty
        return earliest_arrival - required_stability


path = TimingPath(
    clock_period=2.0,
    clock_to_q_max=0.12,
    clock_to_q_min=0.05,
    combinational_max=1.55,
    combinational_min=0.18,
    setup=0.10,
    hold=0.06,
    skew=0.04,
    setup_uncertainty=0.08,
    hold_uncertainty=0.02,
)

assert round(path.setup_slack(), 2) == 0.19
assert round(path.hold_slack(), 2) == 0.11
print("setup slack:", path.setup_slack(), "ns")
print("hold slack:", path.hold_slack(), "ns")
~~~

</details>

#### **Clock Skew and Metastability** {#clock-skew-and-metastability}

The clock reaches thousands or millions of storage elements through a physical distribution network. Differences in wire length, buffering, load, and variation cause **clock skew**: corresponding edges do not arrive everywhere at exactly the same time.

![Positive clock skew means the active edge reaches the capture register later than the launch register.](assets/clock-skew.svg){fig-align="center" width="92%"}

With the convention $t_{skew}=t_{capture}-t_{launch}$, positive skew can help setup because the receiving register samples later, but it hurts hold because the old value must survive longer. Negative skew has the opposite effect. Designers use balanced clock trees or clock meshes, characterize variation, and reserve uncertainty. Useful skew may be intentionally introduced, but improving one path can worsen another.

**Clock jitter** differs from skew. Skew is a spatial difference between clock endpoints; jitter is temporal variation of an edge from its expected time across cycles. Both reduce dependable timing margin.

**Metastability** occurs when a bistable element is asked to decide while its input violates the sampling window. Instead of immediately resolving to a valid 0 or 1, its internal voltage can linger near an unstable equilibrium for an unpredictable time. It will eventually resolve, but digital logic cannot guarantee which value wins or exactly when.

Metastability is especially relevant for asynchronous inputs and clock-domain crossings. No finite setup/hold rule can be guaranteed between unrelated clocks. A common solution for a single-bit level is a two-flip-flop synchronizer:

![The first synchronizer stage may become metastable; the second samples one cycle later, greatly reducing the probability that instability reaches destination logic.](assets/metastability-synchronizer.svg){fig-align="center" width="96%"}

The first stage receives the asynchronous signal and may become metastable. The second stage receives the first output after nearly one destination-clock period of resolution time. This does not eliminate metastability; it makes propagation sufficiently improbable for a reliability target.

A simplified relationship is

$$
MTBF\propto\frac{e^{T_{resolve}/\tau}}{f_{clk}f_{data}}.
$$

- $MTBF$ is mean time between observable synchronization failures.
- $T_{resolve}$ is time allowed for the first stage to settle before the next samples it.
- $\tau$ characterizes how quickly the particular flip-flop resolves metastability.
- $f_{clk}$ is destination sampling frequency.
- $f_{data}$ is asynchronous transition frequency.
- The exponential means a modest increase in resolution time can improve reliability dramatically.

Exact library models include a device-dependent constant and measured parameters. The formula is a reliability model, not a promise that two stages are always enough.

Multi-bit values require more than placing an independent synchronizer on every bit: bits could resolve on different cycles and form a word that never existed. Handshakes, Gray-coded counters, asynchronous FIFOs, or source-synchronous protocols preserve coherence. Narrow pulses may also disappear between destination edges, so pulse stretching or toggle/handshake encoding is needed.

| Crossing | Typical mechanism | Reason |
|---|---|---|
| one stable control level | two-flip-flop synchronizer | contains metastability probability |
| one-cycle pulse | pulse stretch, toggle, or handshake | guarantees destination observes event |
| changing multi-bit bus | handshake with stable data | transfers one coherent word |
| continuous stream | asynchronous FIFO with Gray pointers | separates write and read clock domains |

Metastability cannot be simulated faithfully as a deterministic digital value. Verification instead checks structural crossing rules and protocols, while reliability analysis uses characterized timing parameters.

### **Combinational and Sequential Logic Compared** {#combinational-and-sequential-logic-compared}

Combinational and sequential logic are not competing alternatives. A useful synchronous system deliberately alternates them: registers preserve state, combinational blocks calculate next state and outputs, and clock edges commit selected results.

| Dimension | Combinational logic | Sequential logic |
|---|---|---|
| dependence | current input only | current input plus stored state |
| mathematical form | $Y=F(X)$ | $S^{+}=F(S,X)$ and $Y=G(S,X)$ |
| memory | none | latches, flip-flops, registers, arrays |
| feedback | normally avoided | controlled feedback creates storage |
| time model | settles after input change | advances at level/edge events |
| main correctness checks | truth table, equivalence, critical path | state transitions, reset, setup, hold, CDC |
| examples | adder, MUX, decoder, ALU | counter, FSM, register file, cache state |

A clean design keeps these roles visually and behaviorally distinct. Clocked processes update storage; combinational processes calculate candidates. Every state variable has a reset or initialization story. Every transition has defined behavior. Every register-to-register path passes both maximum-delay and minimum-delay checks, and every asynchronous crossing uses an appropriate protocol.

The central abstraction is the register-transfer step:

$$
\text{old state}\;\longrightarrow\;\text{combinational transformation}\;\longrightarrow\;\text{new state}.
$$

Within a cycle, many gates evaluate concurrently. At the active edge, many registers capture concurrently. This combination of spatial parallelism and discrete state updates is what lets a processor be both a physical circuit and a predictable abstract machine.

**Chapter summary.** State compresses relevant history so identical inputs can produce context-dependent behavior. Cross-coupled gates create bistable storage; SR latches expose set/reset feedback, D latches provide level-sensitive data storage, and D flip-flops provide edge-triggered sampling. Registers collect bits into words, counters generate state sequences, and shift registers move data between stages. Finite-state machines turn remembered situations into controlled transitions, with Moore outputs depending on state and Mealy outputs also depending on current input. Register files add addressed multiport access, while ROM, SRAM, and DRAM trade programmability, speed, density, refresh, and persistence through different cell mechanisms. Finally, propagation and contamination delay, setup and hold, skew, jitter, and metastability determine whether logically correct state transitions are physically reliable. Chapter 05 uses these storage and datapath foundations to explain how an instruction set exposes machine operations to software.